# Neural Network (Rede Neural)

## Table of Contents
* [Libraries](#Libraries-(Bibliotecas))
* [Leitura e Preparação dos Dados](#Leitura-e-Preparação-dos-Dados)
* [Buscando os Melhores Hiperparâmetros](#Buscando-os-Melhores-Hiperparâmetros)
* [Rede Neural](#Rede-Neural)
* [Análise de Erro](#Análise-de-Erro)
* [Retreinando o Modelo](#Retreinando-o-Modelo)
* [Criando o Resultado](#Criando-o-Resultado-das-Previsões-para-Importar-ao-Kaggle)

## Libraries (Bibliotecas)

In [4]:
import pandas as pd
import numpy as np
%matplotlib inline
import tensorflow as tf
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from skopt import gp_minimize
np.set_printoptions(precision=2)
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(1)

## Leitura e Preparação dos Dados

In [6]:
train = pd.read_csv('../dataset/train.csv')
test = pd.read_csv('../dataset/test.csv')
print(f'Train set shape: {train.shape}\nTest set shape: {test.shape}')

Train set shape: (891, 12)
Test set shape: (418, 11)


In [7]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
train['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [9]:
train['Sex_b'] = train['Sex'].map(lambda x: 1 if x == 'female' else 0)
test['Sex_b'] = test['Sex'].map(lambda x: 1 if x == 'female' else 0)

In [10]:
train['Sex_b'].value_counts()

Sex_b
0    577
1    314
Name: count, dtype: int64

In [11]:
train['Ticket']

0             A/5 21171
1              PC 17599
2      STON/O2. 3101282
3                113803
4                373450
             ...       
886              211536
887              112053
888          W./C. 6607
889              111369
890              370376
Name: Ticket, Length: 891, dtype: object

In [12]:
train['Name'].value_counts()

Name
Dooley, Mr. Patrick                                    1
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
                                                      ..
Hewlett, Mrs. (Mary D Kingcome)                        1
Vestrom, Miss. Hulda Amanda Adolfina                   1
Andersson, Mr. Anders Johan                            1
Saundercock, Mr. William Henry                         1
Bonnell, Miss. Elizabeth                               1
Name: count, Length: 891, dtype: int64

In [13]:
train['Embarked_S'] = (train['Embarked'] == 'S').astype(int)
train['Embarked_C'] = (train['Embarked'] == 'C').astype(int)

train['Cabin_null'] = train['Cabin'].isnull().astype(int)
train['Cabin_C'] = train['Cabin'].fillna('').str.count('C').astype(int)
train['Cabin_E'] = train['Cabin'].fillna('').str.count('E').astype(int)
train['Cabin_G'] = train['Cabin'].fillna('').str.count('G').astype(int)
train['Cabin_D'] = train['Cabin'].fillna('').str.count('D').astype(int)
train['Cabin_A'] = train['Cabin'].fillna('').str.count('A').astype(int)
train['Cabin_B'] = train['Cabin'].fillna('').str.count('B').astype(int)
train['Cabin_F'] = train['Cabin'].fillna('').str.count('F').astype(int)
train['Cabin_T'] = train['Cabin'].fillna('').str.count('T').astype(int)

train['Name_Miss'] = train['Name'].str.contains('Miss.').astype(int)
train['Name_Mrs'] = train['Name'].str.contains('Mrs.').astype(int)
train['Name_Master'] = train['Name'].str.contains('Master.').astype(int)
train['Name_Col'] = train['Name'].str.contains('Col.').astype(int)
train['Name_Major'] = train['Name'].str.contains('Major.').astype(int)
train['Name_Mr'] = train['Name'].str.contains('Mr.').astype(int)
train['Name_Dr'] = train['Name'].str.contains('Dr.').astype(int)
train['Name_Don'] = train['Name'].str.contains('Don.').astype(int)
train['Name_Sir'] = train['Name'].str.contains('Sir.').astype(int)

train['Ticket_cat'] = train['Ticket'].map(lambda x: list(train['Ticket'].unique()).index(x) if x in list(train['Ticket'].unique()) else 0)
train['Ticket_num'] = train['Ticket'].str.split().map(lambda x: x[1] if len(x) > 1 and x[1].isnumeric() else x[0] if x[0].isnumeric() else 0)

In [14]:
test['Embarked_S'] = (test['Embarked'] == 'S').astype(int)
test['Embarked_C'] = (test['Embarked'] == 'C').astype(int)

test['Cabin_null'] = test['Cabin'].isnull().astype(int)
test['Cabin_C'] = test['Cabin'].fillna('').str.count('C').astype(int)
test['Cabin_E'] = test['Cabin'].fillna('').str.count('E').astype(int)
test['Cabin_G'] = test['Cabin'].fillna('').str.count('G').astype(int)
test['Cabin_D'] = test['Cabin'].fillna('').str.count('D').astype(int)
test['Cabin_A'] = test['Cabin'].fillna('').str.count('A').astype(int)
test['Cabin_B'] = test['Cabin'].fillna('').str.count('B').astype(int)
test['Cabin_F'] = test['Cabin'].fillna('').str.count('F').astype(int)
test['Cabin_T'] = test['Cabin'].fillna('').str.count('T').astype(int)

test['Name_Miss'] = test['Name'].str.contains('Miss.').astype(int)
test['Name_Mrs'] = test['Name'].str.contains('Mrs.').astype(int)
test['Name_Master'] = test['Name'].str.contains('Master.').astype(int)
test['Name_Col'] = test['Name'].str.contains('Col.').astype(int)
test['Name_Major'] = test['Name'].str.contains('Major.').astype(int)
test['Name_Mr'] = test['Name'].str.contains('Mr.').astype(int)
test['Name_Dr'] = test['Name'].str.contains('Dr.').astype(int)
test['Name_Don'] = test['Name'].str.contains('Don.').astype(int)
test['Name_Sir'] = test['Name'].str.contains('Sir.').astype(int)

test['Ticket_cat'] = test['Ticket'].map(lambda x: list(test['Ticket'].unique()).index(x) if x in list(test['Ticket'].unique()) else 0)
test['Ticket_num'] = test['Ticket'].str.split().map(lambda x: x[1] if len(x) > 1 and x[1].isnumeric() else x[0] if x[0].isnumeric() else 0)

In [15]:
variaveis = ['Sex_b', 'Age', 'Pclass', 'Embarked_S', 'Embarked_C', 'SibSp', 'Parch', 'Fare', 'Cabin_null',
             'Cabin_C', 'Cabin_E', 'Cabin_G', 'Cabin_D', 'Cabin_A', 'Cabin_B', 'Cabin_F', 'Cabin_T',
             'Name_Miss', 'Name_Mrs', 'Name_Master', 'Name_Col', 'Name_Major', 'Name_Mr', 'Name_Dr', 'Name_Don',
             'Name_Sir', 'Ticket_num', 'Ticket_cat']

In [126]:
X = train[variaveis].fillna(-1).copy()
X_test = test[variaveis].fillna(-1).copy()
y = train['Survived'].to_numpy().reshape((-1, 1)).copy()

In [105]:
#X = pd.get_dummies(data=X, prefix=['Ticket_cat'], columns=['Ticket_cat'], dtype=int).drop(columns=['Ticket_cat_0'], axis=0)
X

,Sex_b,Age,Pclass,Embarked_S,Embarked_C,SibSp,Parch,Fare,Cabin_null,Cabin_C,...,Name_Mrs,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir,Ticket_num,Ticket_cat
0,0,22.0,3,1,0,1,0,7.2500,1,0,...,0,0,0,0,1,0,0,0,21171,0
1,1,38.0,1,0,1,1,0,71.2833,0,1,...,1,0,0,0,1,0,0,0,17599,1
2,1,26.0,3,1,0,0,0,7.9250,1,0,...,0,0,0,0,0,0,0,0,3101282,2
3,1,35.0,1,1,0,1,0,53.1000,0,1,...,1,0,0,0,1,0,0,0,113803,3
4,0,35.0,3,1,0,0,0,8.0500,1,0,...,0,0,0,0,1,0,0,0,373450,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,27.0,2,1,0,0,0,13.0000,1,0,...,0,0,0,0,0,0,0,0,211536,677
887,1,19.0,1,1,0,0,0,30.0000,0,0,...,0,0,0,0,0,0,0,0,112053,678
888,1,-1.0,3,1,0,1,2,23.4500,1,0,...,0,0,0,0,0,0,0,0,6607,614
889,0,26.0,1,0,1,0,0,30.0000,0,1,...,0,0,0,0,1,0,0,0,111369,679


In [107]:
#X_test = pd.get_dummies(data=X_test, prefix=['Ticket_cat'], columns=['Ticket_cat'], dtype=int).drop(columns=['Ticket_cat_0'], axis=0)
X_test

,Sex_b,Age,Pclass,Embarked_S,Embarked_C,SibSp,Parch,Fare,Cabin_null,Cabin_C,...,Name_Mrs,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir,Ticket_num,Ticket_cat
0,0,34.5,3,0,0,0,0,7.8292,1,0,...,0,0,0,0,1,0,0,0,330911,0
1,1,47.0,3,1,0,1,0,7.0000,1,0,...,1,0,0,0,1,0,0,0,363272,1
2,0,62.0,2,0,0,0,0,9.6875,1,0,...,0,0,0,0,1,0,0,0,240276,2
3,0,27.0,3,1,0,0,0,8.6625,1,0,...,0,0,0,0,1,0,0,0,315154,3
4,1,22.0,3,1,0,1,1,12.2875,1,0,...,1,0,0,0,1,0,0,0,3101298,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,0,-1.0,3,1,0,0,0,8.0500,1,0,...,0,0,0,0,1,0,0,0,3236,358
414,1,39.0,1,0,1,0,0,108.9000,0,1,...,0,0,0,0,0,0,1,0,17758,359
415,0,38.5,3,1,0,0,0,7.2500,1,0,...,0,0,0,0,1,0,0,0,3101262,360
416,0,-1.0,3,1,0,0,0,8.0500,1,0,...,0,0,0,0,1,0,0,0,359309,361


## Buscando os Melhores Hiperparâmetros
Utilizando bayesian optimization

In [110]:
def fit_model(hparams):
    threshold = .5
    learning_rate = hparams[0]
    lambda_r = hparams[1]
    l1 = hparams[2]
    l2 = hparams[3]
    l3 = hparams[4]
    l4 = hparams[5]
    
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=hparams[2], activation='relu', name='L1', kernel_regularizer=tf.keras.regularizers.l2(hparams[1])),
        tf.keras.layers.Dense(units=hparams[3], activation='relu', name='L2', kernel_regularizer=tf.keras.regularizers.l2(hparams[1])),
        tf.keras.layers.Dense(units=hparams[4], activation='relu', name='L3', kernel_regularizer=tf.keras.regularizers.l2(hparams[1])),
        tf.keras.layers.Dense(units=hparams[5], activation='relu', name='L4', kernel_regularizer=tf.keras.regularizers.l2(hparams[1])),
        tf.keras.layers.Dense(units=1, activation='linear', name='L5')
    ], name='model')
    
    model.compile(loss=BinaryCrossentropy(from_logits=True),
                  optimizer=Adam(learning_rate=hparams[0]), metrics=['accuracy'])
    
    model.fit(X_opt, y, epochs=200, verbose=0)
    
    yhat = model.predict(X_opt)
    yhat = tf.math.sigmoid(yhat)
    yhat = np.where(yhat >= threshold, 1, 0)
    acc = np.mean(yhat == y)
    
    
    return -acc

In [112]:
space = [
    (1e-6, 1e-1, 'log-uniform'), # learning_rate
    (1e-6, 1e-1), # lambda_r
    (100, 300), # l1
    (75, 200), # l2
    (50, 150), # l3
    (10, 100), # l4
]

X_opt = StandardScaler().fit_transform(X)
opt = gp_minimize(fit_model, space, random_state=42, verbose=0, n_calls=25, n_random_starts=10)
opt.x

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[0.001293563047325189,
 1e-06,
 np.int64(300),
 np.int64(75),
 np.int64(150),
 np.int64(100)]

## Rede Neural

In [130]:
learning_rate = opt.x[0]
lambda_r = opt.x[1]
threshold = 0.5

kf = RepeatedKFold(n_splits=3, n_repeats=1, random_state=42)

scaler = StandardScaler()

step_train = []
step = []

for linhas_train, linhas_cv in kf.split(X):
    X_train, X_cv = X.iloc[linhas_train].copy(), X.iloc[linhas_cv].copy()
    y_train, y_cv = y[linhas_train], y[linhas_cv]

    X_train = scaler.fit_transform(X_train)
    X_cv = scaler.transform(X_cv)

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=opt.x[2], activation='relu', name='L1', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=opt.x[3], activation='relu', name='L2', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=opt.x[4], activation='relu', name='L3', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=opt.x[5], activation='relu', name='L4', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=1, activation='linear', name='L5')
    ], name='model')
    
    model.compile(loss=BinaryCrossentropy(from_logits=True),
                  optimizer=Adam(learning_rate=learning_rate), metrics=['accuracy'])
    
    model.fit(X_train, y_train, epochs=200, verbose=0)

    yhat_train = model.predict(X_train)
    yhat = model.predict(X_cv)
    
    yhat_train = tf.math.sigmoid(yhat_train)
    yhat_train = np.where(yhat_train >= threshold, 1, 0)
    acc_train = np.mean(yhat_train == y_train)

    yhat = tf.math.sigmoid(yhat)
    yhat = np.where(yhat >= threshold, 1, 0)
    acc = np.mean(yhat == y_cv)

    print(f'\nacc_train: {acc_train:.4f}, acc_cv: {acc:.4f}\n')

    step_train.append(acc_train)
    step.append(acc)

print(f'Train mean: {np.mean(step_train):.2f}, CV mean: {np.mean(step):.2f}')

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

acc_train: 0.9714, acc_cv: 0.7811

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

acc_train: 0.9865, acc_cv: 0.7980

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

acc_train: 0.9882, acc_cv: 0.7845

Train mean: 0.98, CV mean: 0.79


In [131]:
model.summary()

Model: "model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ L1 (Dense)                           │ (None, 300)                 │           8,700 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ L2 (Dense)                           │ (None, 75)                  │          22,575 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ L3 (Dense)                           │ (None, 150)                 │          11,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ L4 (Dense)                           │ (None, 100)                 │          15,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ L5 (Dense)                           │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 173,630 (678.25 KB)

 Trainable params: 57,876 (226.08 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 115,754 (452.17 KB)

## Análise de Erro

In [133]:
X_cv_erro = train.iloc[linhas_cv].copy()
X_cv_erro['yhat'] = yhat
X_cv_erro.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir,Ticket_cat,Ticket_num,yhat
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,0,0,0,1,0,0,0,1,17599,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,0,0,0,1,0,0,0,4,373450,0
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,...,0,0,0,1,0,0,0,8,347742,1
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,...,0,0,0,1,0,0,0,13,347082,0
14,15,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,...,0,0,0,0,0,0,0,14,350406,1


In [134]:
erro = X_cv_erro[X_cv_erro['Survived'] != X_cv_erro['yhat']]
erro = erro[['Survived', 'yhat', 'Name', 'Cabin', 'Embarked', 'Sex', 'Sex_b', 'Age', 'Pclass', 'Embarked_S',
             'Embarked_C', 'SibSp', 'Parch', 'Fare', 'Cabin_null',
             'Cabin_C', 'Cabin_E', 'Cabin_G', 'Cabin_D', 'Cabin_A', 'Cabin_B', 'Cabin_F', 'Cabin_T',
             'Name_Miss', 'Name_Mrs', 'Name_Master', 'Name_Col', 'Name_Major', 'Name_Mr', 'Name_Dr', 'Name_Don', 'Name_Sir']]
erro.head()

,Survived,yhat,Name,Cabin,Embarked,Sex,Sex_b,Age,Pclass,Embarked_S,...,Cabin_T,Name_Miss,Name_Mrs,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir
14,0,1,"Vestrom, Miss. Hulda Amanda Adolfina",NaN,S,female,1,14.0,3,1,...,0,1,0,0,0,0,0,0,0,0
21,1,0,"Beesley, Mr. Lawrence",D56,S,male,0,34.0,2,1,...,0,0,0,0,0,0,1,0,0,0
36,1,0,"Mamee, Mr. Hanna",NaN,C,male,0,NaN,3,0,...,0,0,0,0,0,0,1,0,0,0
47,1,0,"O'Driscoll, Miss. Bridget",NaN,Q,female,1,NaN,3,0,...,0,1,0,0,0,0,0,1,0,0
53,1,0,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",NaN,S,female,1,29.0,2,1,...,0,0,1,0,0,0,1,0,0,0


In [135]:
female = erro[erro['Sex_b'] == 1]
male = erro[erro['Sex_b'] == 0]

In [136]:
female.sort_values('Survived')

,Survived,yhat,Name,Cabin,Embarked,Sex,Sex_b,Age,Pclass,Embarked_S,...,Cabin_T,Name_Miss,Name_Mrs,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir
14,0,1,"Vestrom, Miss. Hulda Amanda Adolfina",NaN,S,female,1,14.0,3,1,...,0,1,0,0,0,0,0,0,0,0
111,0,1,"Zabour, Miss. Hileni",NaN,C,female,1,14.5,3,0,...,0,1,0,0,0,0,0,0,0,0
251,0,1,"Strom, Mrs. Wilhelm (Elna Matilda Persson)",G6,S,female,1,29.0,3,1,...,0,0,1,0,0,0,1,0,0,0
205,0,1,"Strom, Miss. Telma Matilda",G6,S,female,1,2.0,3,1,...,0,1,0,0,0,0,0,0,0,0
415,0,1,"Meek, Mrs. Thomas (Annie Louise Rowley)",NaN,S,female,1,NaN,3,1,...,0,0,1,0,0,0,1,0,0,0
419,0,1,"Van Impe, Miss. Catharina",NaN,S,female,1,10.0,3,1,...,0,1,0,0,0,0,0,0,0,0
297,0,1,"Allison, Miss. Helen Loraine",C22 C26,S,female,1,2.0,1,1,...,0,1,0,0,0,0,0,0,0,0
474,0,1,"Strandberg, Miss. Ida Sofia",NaN,S,female,1,22.0,3,1,...,0,1,0,0,0,0,0,0,0,0
564,0,1,"Meanwell, Miss. (Marion Ogden)",NaN,S,female,1,NaN,3,1,...,0,1,0,0,0,0,0,0,0,0
702,0,1,"Barbara, Miss. Saiide",NaN,C,female,1,18.0,3,0,...,0,1,0,0,0,0,0,0,0,0


In [137]:
male.sort_values('Survived')

,Survived,yhat,Name,Cabin,Embarked,Sex,Sex_b,Age,Pclass,Embarked_S,...,Cabin_T,Name_Miss,Name_Mrs,Name_Master,Name_Col,Name_Major,Name_Mr,Name_Dr,Name_Don,Name_Sir
102,0,1,"White, Mr. Richard Frasar",D26,S,male,0,21.0,1,1,...,0,0,0,0,0,0,1,0,0,0
452,0,1,"Foreman, Mr. Benjamin Laventall",C111,C,male,0,30.0,1,0,...,0,0,0,0,0,0,1,0,0,0
295,0,1,"Lewy, Mr. Ervin G",NaN,C,male,0,NaN,1,0,...,0,0,0,0,0,0,1,0,0,0
270,0,1,"Cairns, Mr. Alexander",NaN,S,male,0,NaN,1,1,...,0,0,0,0,0,0,1,0,0,0
636,0,1,"Leinonen, Mr. Antti Gustaf",NaN,S,male,0,32.0,3,1,...,0,0,0,0,0,0,1,0,0,0
671,0,1,"Davidson, Mr. Thornton",B71,S,male,0,31.0,1,1,...,0,0,0,0,0,0,1,0,0,0
672,0,1,"Mitchell, Mr. Henry Michael",NaN,S,male,0,70.0,2,1,...,0,0,0,0,0,0,1,0,0,0
505,0,1,"Penasco y Castellana, Mr. Victor de Satode",C65,C,male,0,18.0,1,0,...,0,0,0,0,0,0,1,0,0,0
769,0,1,"Gronnestad, Mr. Daniel Danielsen",NaN,S,male,0,32.0,3,1,...,0,0,0,0,0,0,1,0,0,0
850,0,1,"Andersson, Master. Sigvard Harald Elias",NaN,S,male,0,4.0,3,1,...,0,0,0,1,0,0,0,0,0,0


## Retreinando o Modelo

In [139]:
scaler = StandardScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

In [140]:
lambda_r = 1e-2
learning_rate = 1e-4

threshold = .5

model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=250, activation='relu', name='L1', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=150, activation='relu', name='L2', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=75, activation='relu', name='L3', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=25, activation='relu', name='L4', kernel_regularizer=tf.keras.regularizers.l2(lambda_r)),
        tf.keras.layers.Dense(units=1, activation='linear', name='L5')
], name='model')
    
model.compile(loss=BinaryCrossentropy(from_logits=True),
                  optimizer=Adam(learning_rate=learning_rate))
    
model.fit(X, y, epochs=200)

Epoch 1/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4.4615
Epoch 2/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.2592  
Epoch 3/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.0949
Epoch 4/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.9349  
Epoch 5/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.7697  
Epoch 6/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.6103
Epoch 7/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.4545  
Epoch 8/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.3069
Epoch 9/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.1722 
Epoch 10/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.0369
Epoch 11/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.9078 
Epoch 12/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.8051 
Epoch 13/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.6748  
Epoch 14/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.6101
Epoch 15/200
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss

In [141]:
# computando a previsão de X_test
y_hat = model.predict(X_test)
y_hat = tf.math.sigmoid(y_hat)
y_hat = np.where(y_hat >=threshold, 1, 0)
y_hat

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


array([[0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
    

## Criando o Resultado das Previsões para Importar ao Kaggle

In [143]:
result = pd.Series(y_hat.reshape(-1), index=test['PassengerId'], name='Survived')
result

PassengerId
892     0
893     0
894     0
895     0
896     1
       ..
1305    0
1306    0
1307    0
1308    0
1309    1
Name: Survived, Length: 418, dtype: int64

In [ ]:
result.to_csv('../yhat/neural_network_model.csv', header=True)